# 3.5 MPI 扩展性分析

## 本节学习目标

- 比较不同进程数
- 识别通信主导区

## 环境检查

直接检查本节需要的运行环境；若检查失败，请先在对应 CPU/NPU 节点加载课程要求的工具链。


In [ ]:
import shutil, subprocess
for tool in ("cmake", "mpicxx", "mpirun"):
    path = shutil.which(tool)
    if path is None:
        raise RuntimeError(f"缺少必需工具：{tool}")
    print(f"{tool}: {path}")


## 扩展性

进程增加会缩短局部行块，但完整 x 仍需广播、y 仍需汇聚。小矩阵固定通信开销占比更高，大矩阵也可能受到内存带宽和网络限制。

## 实验控制

进程集合应依据分配到的 CPU slot 选择，不应使用 oversubscribe 伪造性能。不同进程数必须复用同一矩阵、warmup 和 repeat。

## 预期现象与结果分析

扩展性报告应同时包含端到端、通信、计算和负载均衡。即使 local compute 下降，总时间也可能因通信增长而上升。

## 原工程 16 进程历史实测

`mpi-SpMV/README.md` 记录了远程 16 CPU 服务器、`warmup=10`、`repeat=100` 的六矩阵测试。下表摘录端到端结果；六组 L2 error 均为 0，全部 PASS。

<table style="margin-left: 0; text-align: left;">
  <thead>
    <tr>
      <th>Matrix</th>
      <th>CPU baseline (ms)</th>
      <th>MPI 16 进程 (ms)</th>
      <th>Speedup</th>
      <th>主要观察</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>U1</td>
      <td>3.478545</td>
      <td>1.647440</td>
      <td>2.111x</td>
      <td>小矩阵通信占比较高</td>
    </tr>
    <tr>
      <td>U2</td>
      <td>111.119954</td>
      <td>23.523875</td>
      <td>4.724x</td>
      <td>local SpMV 14.360 ms</td>
    </tr>
    <tr>
      <td>L1</td>
      <td>4.096557</td>
      <td>1.515193</td>
      <td>2.704x</td>
      <td>nnz balance 1.002112</td>
    </tr>
    <tr>
      <td>L2</td>
      <td>103.167710</td>
      <td>22.455029</td>
      <td>4.594x</td>
      <td>local SpMV 13.690 ms</td>
    </tr>
    <tr>
      <td>B1</td>
      <td>2.167312</td>
      <td>1.096582</td>
      <td>1.976x</td>
      <td>固定通信开销明显</td>
    </tr>
    <tr>
      <td>B2</td>
      <td>27.123289</td>
      <td>10.295174</td>
      <td>2.635x</td>
      <td>Bcast 7.153 ms，高于计算 1.831 ms</td>
    </tr>
  </tbody>
</table>

阶段时间分别做 `MPI_MAX`，不能相加复算端到端时间。该历史结果说明：累计 nnz 分区把 balance ratio 控制在接近 1，但通信仍会限制扩展性。实际数值会随 MPI 实现、绑核和系统负载变化。

## 课后实践

设计不超过可用 CPU slot 的进程数实验，并说明选择依据。

参考答案见 `answer/03.05_answer.md`。